# Ingesta archivo drivers.json 
1. Leer el archivo usando API spark dataframe reader 
1. Definir un esquema (preservar la estructura anidada)
1. Agregar columnas metadata 
    - Archivo fuente
    - Ingesta Timestamp
1. Escribir a la tabla bronze delta   

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/02.Ayuda_Bronze

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
# Definir el nombre  de la tabla y archivo fuente 
archivo_fuente = f"{ruta}/drivers.json"
nombre_tabla = f"{catalogo}.{esquema_bronze}.drivers"

#### 1 - Leer el archivo json usando API dataframe reader

In [0]:
# Definir el esquema
from pyspark.sql.types import StructType, StructField, StringType, DateType

esquema_name = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

esquema_drivers = StructType([
    StructField('driverId', StringType()),
    StructField('name',esquema_name),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])

In [0]:
# Leer la data del archivo drivers
df_drivers = (
    spark.read
       .format('json')
       .schema(esquema_drivers)
       .option('mode', 'FAILFAST')
       .load(archivo_fuente)
)

#### 2 - Agregar columnas metadata
- Archivo Fuente
- Ingesta Timestamp

In [0]:
df_final_drivers  = add_ingesta_metadata(df_drivers )

#### 3 - Escribir  al atabla bronze delta

In [0]:
(
    df_final_drivers 
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(nombre_tabla)
)

In [0]:
display(spark.table(nombre_tabla))